# Notebook de référence — Examen Machine Learning IMDS3

**But :** retrouver rapidement, pour chaque question d'examen, le bon bout de code (uniquement
les fonctions vues dans les TP) ou la bonne réponse théorique.

**Organisation :**
1. Réponses aux questions **théoriques** récurrentes
2. **Boîte à outils** : briques réutilisables (prétraitement, activations, coûts, backprop, entraînement…)
3. **Scénarios complets** exécutables (un par type de problème), chacun annoté avec la question d'examen correspondante :
   - A. Classification binaire (sigmoïde, sortie 2D) — TP4
   - B. Classification multiclasse (softmax) + régularisation + train/val/test + tableau d'hyperparamètres — TP5/TP7, *session 2024 Q3*
   - C. Régression (réseau de neurones, sortie linéaire) — *examen 2025 Q1–3*
   - D. Validation croisée (moyenne de 5 modèles) — *examen 2025 Q4–5*
   - E. Régression logistique régularisée — *session 2024 Q5*
   - F. K-means — TP8
   - G. PCA — TP8
   - H. Biais-variance : courbes d'apprentissage et de validation — TP6

> Chaque scénario est **autonome** : il génère ses propres données synthétiques, définit ses fonctions
> et tourne tout seul. À l'examen, repère le scénario qui ressemble à ton sujet, copie la cellule,
> et remplace les données synthétiques par celles fournies.

**Corrections par rapport aux TP :** softmax stable (`exp(z - max(z))`) et régularisation correcte
(`+ (λ/m)·W` et **non** `np.sum(W)`).


In [ ]:
# Imports communs (tous ceux croisés dans les TP)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import optimize          # TP6 : optimize.minimize
from scipy.io import loadmat        # TP5/TP6/TP8 : lecture de fichiers .mat
from sklearn.utils import shuffle   # mélange des données
from sklearn.preprocessing import LabelEncoder  # encodage des variables catégorielles

## 1. Réponses aux questions théoriques récurrentes

**Dimensions des matrices** (X de taille `(n1, m)`, colonnes = observations, `n2` neurones cachés,
`n3` sorties) :

| Objet | Dimension |
|---|---|
| `X = a(1)` | `(n1, m)` |
| `W2` | `(n2, n1)` |
| `b2` | `(n2, 1)` |
| `z2 = a2` | `(n2, m)` |
| `W3` | `(n3, n2)` |
| `b3` | `(n3, 1)` |
| `z3 = a3` | `(n3, m)` |

*Régression :* `n3 = 1`. *Classification binaire (2 sorties) :* `n3 = 2`. *Multiclasse :* `n3 = nb de classes`.

**Pourquoi mélanger (shuffle) les données ?** Parce qu'elles sont souvent rangées par classe
(les 60 sinus puis les 60 exp, les chiffres triés…). Sans mélange, le découpage train/test mettrait
certaines classes uniquement dans le test : le modèle n'apprendrait jamais à les reconnaître et
l'évaluation serait biaisée.

**Pourquoi 3 jeux (train / validation / test) et rôle de chacun ?**
- **Train** : sert à trouver les *paramètres* du modèle (W2, W3, b2, b3) par descente de gradient.
- **Validation** : sert à choisir les *hyperparamètres* (n2, α, λ, Niter, nb de couches) en comparant
  les performances. On ne l'utilise pas pour la descente de gradient.
- **Test** : utilisé une seule fois, à la toute fin, pour estimer la performance réelle du modèle final
  sur des données jamais vues.

**Pourquoi pas de validation quand les hyperparamètres sont fixés ?** (examen 2025 Q3, session 2024)
Le jeu de validation ne sert qu'à *choisir* les hyperparamètres. Si l'énoncé les impose
(n2, α, Niter, λ donnés), il n'y a rien à sélectionner : le jeu de validation devient inutile.

**Les deux opérations standard de prétraitement** (session 2024 Q1) : **mélanger** les données et
les **normaliser** (souvent précédées de l'encodage des variables catégorielles et du traitement des
valeurs manquantes).

**Sigmoïde vs softmax en sortie ?** La sigmoïde donne des sorties indépendantes dans `[0,1]`. La softmax
donne un **vecteur de probabilités** qui somme à 1 — adaptée à une classification où les classes
sont mutuellement exclusives.

**delta_3 = a3 − y dans les deux cas.** Avec entropie croisée + sigmoïde **ou** entropie croisée + softmax,
le calcul de `delta_3` se simplifie en `a3 − y`. C'est la simplification clé des TP4/TP5 : il n'y a
**aucune différence** dans le calcul de `delta_3` entre sigmoïde et softmax (avec entropie croisée).

**Régularisation L2.** On ajoute `λ/(2m)·Σ W²` au coût. Effet sur le gradient : `+ (λ/m)·W` sur les
**poids uniquement** (les biais ne sont pas régularisés). Elle limite le surapprentissage en empêchant
les poids de devenir trop grands.


## 2. Boîte à outils (briques réutilisables)

Chaque cellule ci-dessous est un petit exemple exécutable d'une opération que tu auras à faire.


### 2.1 Lecture & prétraitement d'un CSV (pandas, valeurs manquantes, encodage)

In [ ]:
# --- Lecture d'un CSV (séparateur virgule, 1re ligne = en-têtes) ---
# data = pd.read_table('fichier.csv', sep=',', header=0)   # variante TP météo
# df   = pd.read_csv('fichier.csv')                         # variante examen 2025

# Démo sur un petit DataFrame synthétique :
df = pd.DataFrame({
    "id":[1,2,3,4],
    "gender":["M","F","F","M"],
    "diet":["good","poor",None,"good"],
    "hours":[5.0,2.0,3.0,4.0],
    "score":[80,55,60,75],
})

# Supprimer une colonne inutile (ex : student_id) :
df = df.drop("id", axis=1)

# Remplacer les valeurs manquantes par la valeur la plus fréquente (mode) :
df["diet"] = df["diet"].fillna(df["diet"].mode()[0])

# Encoder les colonnes catégorielles (texte -> entiers) :
le = LabelEncoder()
for col in ["gender", "diet"]:
    df[col] = le.fit_transform(df[col])

print(df)

### 2.2 One-hot encoding du vecteur de labels (TP5)

In [ ]:
# y_lab : labels entiers (0..n3-1), longueur m  ->  Y : matrice (n3, m)
y_lab = np.array([0, 2, 1, 3, 0])
n3 = 4
m  = y_lab.size

Y = np.zeros((n3, m))
for i in range(m):
    Y[y_lab[i], i] = 1.0

print(Y)

### 2.3 Mélange des données (shuffle)

In [ ]:
# Mélange en gardant la correspondance X <-> labels. random_state pour la reproductibilité.
X_demo = np.arange(20).reshape(4, 5).astype(float)   # (n1=4, m=5)
lab    = np.array([0, 1, 2, 3, 0])

# shuffle travaille sur les LIGNES -> on transpose pour mélanger les colonnes (observations)
Xs, labs = shuffle(X_demo.T, lab, random_state=42)
Xs = Xs.T
print("X mélangé :\n", Xs, "\nlabels :", labs)

### 2.4 Découpage train/test (2 jeux) et train/val/test (3 jeux)

In [ ]:
# Données rangées en COLONNES : X de taille (n1, m). On découpe sur les colonnes.
m_total = 500
X  = np.random.randn(7, m_total)
Y  = np.random.randn(4, m_total)
yl = np.random.randint(0, 4, m_total)

# --- 2 jeux : 900/100 dans l'examen 2025 (ici 400/100 pour la démo) ---
m_train = 400
X_train, X_test = X[:, :m_train], X[:, m_train:]
y_train, y_test = yl[:m_train],   yl[m_train:]

# --- 3 jeux : train/val/test (ex : 3000/1000/1000 au TP7, ici 300/100/100) ---
m_tr, m_va = 300, 100
X_tr = X[:, :m_tr];                 Y_tr = Y[:, :m_tr];                 y_tr = yl[:m_tr]
X_va = X[:, m_tr:m_tr+m_va];        Y_va = Y[:, m_tr:m_tr+m_va];        y_va = yl[m_tr:m_tr+m_va]
X_te = X[:, m_tr+m_va:];            Y_te = Y[:, m_tr+m_va:];            y_te = yl[m_tr+m_va:]
print(X_tr.shape, X_va.shape, X_te.shape)

### 2.5 Normalisation min-max dans [0,1] (paramètres calculés sur le TRAIN)

In [ ]:
# IMPORTANT : on calcule min/max sur le TRAIN puis on applique au TEST (pas l'inverse).
X_train = np.random.rand(5, 100) * 10
X_test  = np.random.rand(5, 30)  * 10

mn = np.min(X_train, axis=1)   # min par variable (par ligne)
mx = np.max(X_train, axis=1)

X_train = ((X_train.T - mn) / (mx - mn)).T
X_test  = ((X_test.T  - mn) / (mx - mn)).T   # mêmes mn/mx que le train
print("train:", X_train.min(), X_train.max())

### 2.6 Fonctions d'activation (sigmoïde, sa dérivée, softmax stable)

In [ ]:
def sigma(z):
    return 1 / (1 + np.exp(-z))

def sigma_prim(z):
    return np.exp(-z) / ((1 + np.exp(-z))**2)

def softmax(z):
    # version stable : on retire le max par colonne pour éviter l'overflow de exp
    exp_z = np.exp(z - np.max(z, axis=0))
    return exp_z / np.sum(exp_z, axis=0)

print(np.round(softmax(np.array([[1.,2.],[3.,0.5]])), 3))

### 2.7 Initialisation aléatoire des poids

In [ ]:
n1, n2, n3 = 400, 25, 10
W2 = np.random.randn(n2, n1) * 0.1   # * 0.1 conseillé en grande dimension (sinon overflow)
b2 = np.zeros((n2, 1))
W3 = np.random.randn(n3, n2) * 0.1
b3 = np.zeros((n3, 1))
print(W2.shape, b2.shape, W3.shape, b3.shape)

### 2.8 Fonctions de coût (les 3 variantes des TP)

In [ ]:
# (a) MSE pour la RÉGRESSION (examen 2025 / notebook_fourni)
def cost_mse(y, y_pred):
    return np.linalg.norm(y - y_pred)**2 / np.size(y)

# (b) Entropie croisée pour la CLASSIFICATION softmax (TP5)
def cost_ce(y, y_pred):
    m = y.shape[1]
    return (-1/m) * np.sum(y * np.log(y_pred + 1e-12))

# (c) Entropie croisée RÉGULARISÉE (TP7) : on pénalise les poids
def cost_ce_reg(y, y_pred, W2, W3, lam):
    m = y.shape[1]
    C  = (-1/m) * np.sum(y * np.log(y_pred + 1e-12))
    C += (lam/(2*m)) * (np.sum(W2**2) + np.sum(W3**2))
    return C
print("ok")

### 2.9 Backpropagation (compute_grad) — versions régression et classification

In [ ]:
# --- CLASSIFICATION (softmax en sortie) + régularisation ---
# delta_3 = A3 - Y aussi bien pour sigmoïde+CE que softmax+CE.
def compute_grad_classif(x, y, W2, W3, b2, b3, lam=0.0):
    m  = x.shape[1]
    A1 = x
    Z2 = W2@A1 + b2;  A2 = sigma(Z2)
    Z3 = W3@A2 + b3;  A3 = softmax(Z3)
    delta_3 = A3 - y
    delta_2 = (W3.T @ delta_3) * sigma_prim(Z2)
    dC_W2 = (1/m)*delta_2@A1.T + (lam/m)*W2     # + (lam/m)*W  : régul. correcte
    dC_W3 = (1/m)*delta_3@A2.T + (lam/m)*W3
    dC_b2 = (1/m)*np.sum(delta_2, axis=1).reshape((W2.shape[0], 1))   # biais NON régularisés
    dC_b3 = (1/m)*np.sum(delta_3, axis=1).reshape((W3.shape[0], 1))
    return dC_W2, dC_W3, dC_b2, dC_b3

# --- RÉGRESSION (sortie linéaire a3 = z3) ---
def compute_grad_reg(x, y, W2, W3, b2, b3):
    m  = x.shape[1]
    A1 = x
    Z2 = W2@A1 + b2;  A2 = sigma(Z2)
    A3 = W3@A2 + b3                # PAS de fonction d'activation en sortie
    delta_3 = A3 - y
    delta_2 = (W3.T @ delta_3) * sigma_prim(Z2)
    dC_W2 = (1/m)*delta_2@A1.T
    dC_W3 = (1/m)*delta_3@A2.T
    dC_b2 = (1/m)*np.sum(delta_2, axis=1).reshape((W2.shape[0], 1))
    dC_b3 = (1/m)*np.sum(delta_3, axis=1).reshape((W3.shape[0], 1))
    return dC_W2, dC_W3, dC_b2, dC_b3
print("ok")

### 2.10 model_predict (classification et régression) + accuracy

In [ ]:
def model_predict_classif(X, W2, W3, b2, b3):
    A2 = sigma(W2@X + b2)
    A3 = softmax(W3@A2 + b3)     # softmax -> vecteur de probabilités
    return A3

def model_predict_reg(X, W2, W3, b2, b3):
    A2 = sigma(W2@X + b2)
    A3 = W3@A2 + b3              # sortie linéaire
    return A3

def accuracy(X, W2, W3, b2, b3, labels):
    A3 = model_predict_classif(X, W2, W3, b2, b3)
    return 100 * np.mean(np.argmax(A3, axis=0) == labels)
print("ok")

### 2.11 Boucle d'entraînement générique (descente de gradient)

In [ ]:
# Squelette à adapter : choisir compute_grad_classif / compute_grad_reg et le bon coût.
def entrainer(X, Y, n1, n2, n3, alpha, Niter, lam=0.0, mode="classif", seed=1):
    np.random.seed(seed)
    W2 = np.random.randn(n2, n1) * 0.1; b2 = np.zeros((n2, 1))
    W3 = np.random.randn(n3, n2) * 0.1; b3 = np.zeros((n3, 1))
    Cost = np.zeros(Niter)
    for j in range(Niter):
        if mode == "classif":
            yp = model_predict_classif(X, W2, W3, b2, b3)
            Cost[j] = cost_ce_reg(Y, yp, W2, W3, lam)
            g = compute_grad_classif(X, Y, W2, W3, b2, b3, lam)
        else:  # régression
            yp = model_predict_reg(X, W2, W3, b2, b3)
            Cost[j] = cost_mse(Y, yp)
            g = compute_grad_reg(X, Y, W2, W3, b2, b3)
        W2 -= alpha*g[0]; W3 -= alpha*g[1]; b2 -= alpha*g[2]; b3 -= alpha*g[3]
    return W2, W3, b2, b3, Cost
print("ok")

## 3. Scénarios complets (autonomes et exécutables)

Chaque scénario génère ses propres données, (re)définit ses fonctions et tourne seul.


### A. Classification binaire — sigmoïde, sortie 2D (TP4)

Distinguer deux familles de données (ici sinus vs exponentielle). Sortie de dimension 2,
décision par `argmax`. Coût : entropie croisée binaire. `delta_3 = a3 − y`.


In [ ]:
# ----- données (TP4) -----
m = 120; n2 = 15; alpha = 0.3; Niter = 8000
np.random.seed(2023)
U = np.random.rand(1, m)
X = np.zeros((1, m))
X[0, :m//2] = np.sin(2*np.pi*U[0, :m//2])   # 60 sinus
X[0, m//2:] = np.exp(U[0, m//2:]) - 1        # 60 exponentielles
Y = np.zeros((2, m))
Y[0, :m//2] = 1.0    # (1,0) = sinus
Y[1, m//2:] = 1.0    # (0,1) = exponentielle

# ----- fonctions -----
def sigma(z): return 1/(1+np.exp(-z))
def sigma_prim(z): return np.exp(-z)/((1+np.exp(-z))**2)
def model_predict(X, W2, W3, b2, b3):
    A2 = sigma(W2@X + b2); A3 = sigma(W3@A2 + b3); return A3
def cost(Y, Yp):
    mm = Y.shape[1]
    return (-1/mm)*np.sum(Y*np.log(Yp) + (1-Y)*np.log(1-Yp))
def compute_grad(x, y, W2, W3, b2, b3):
    A1 = x; Z2 = W2@A1+b2; A2 = sigma(Z2); Z3 = W3@A2+b3; A3 = sigma(Z3)
    d3 = A3 - y                       # simplification entropie croisée
    d2 = (W3.T@d3)*sigma_prim(Z2)
    return ((1/m)*d2@A1.T, (1/m)*d3@A2.T,
            (1/m)*np.sum(d2,axis=1).reshape((n2,1)), (1/m)*np.sum(d3,axis=1).reshape((2,1)))

# ----- init (n3 = 2 sorties) -----
W2 = np.random.randn(n2,1); b2 = np.random.randn(n2,1)
W3 = np.random.randn(2,n2); b3 = np.random.randn(2,1)

# ----- entraînement -----
Cost = np.zeros(Niter)
for j in range(Niter):
    Cost[j] = cost(Y, model_predict(X,W2,W3,b2,b3))
    g = compute_grad(X,Y,W2,W3,b2,b3)
    W2-=alpha*g[0]; W3-=alpha*g[1]; b2-=alpha*g[2]; b3-=alpha*g[3]

# ----- accuracy : décision = argmax des 2 composantes -----
yp = model_predict(X,W2,W3,b2,b3)
acc = 100*np.mean(np.argmax(yp,axis=0) == np.argmax(Y,axis=0))
print("Accuracy train :", round(acc,2), "%")
plt.plot(Cost); plt.xlabel("itérations"); plt.ylabel("coût"); plt.title("A - coût"); plt.show()

### B. Classification multiclasse — softmax + régularisation + train/val/test + tableau (TP5/TP7, session 2024 Q3)

Le cœur de l'examen de classification. On parcourt un tableau d'hyperparamètres `(n2, Niter)`
en mesurant l'accuracy sur la **validation**, on garde le meilleur, puis on évalue une seule fois
sur le **test**.


In [ ]:
# ----- données synthétiques (n1 variables, n3 classes) : à remplacer par les données fournies -----
np.random.seed(0)
n1, n3 = 7, 4
m_total = 600
centres = np.random.randn(n3, n1) * 3
Xall = np.zeros((n1, m_total)); ylab = np.zeros(m_total, dtype=int)
for i in range(m_total):
    c = i % n3; ylab[i] = c
    Xall[:, i] = centres[c] + np.random.randn(n1)

# mélange + one-hot
Xall, ylab = shuffle(Xall.T, ylab, random_state=42); Xall = Xall.T
Y = np.zeros((n3, m_total))
for i in range(m_total): Y[ylab[i], i] = 1.0

# normalisation min-max sur le train (les colonnes train seront [:m_tr])
m_tr, m_va = 360, 120
mn = np.min(Xall[:, :m_tr], axis=1); mx = np.max(Xall[:, :m_tr], axis=1)
Xall = ((Xall.T - mn) / (mx - mn + 1e-12)).T

X_tr, Y_tr, y_tr = Xall[:, :m_tr],           Y[:, :m_tr],           ylab[:m_tr]
X_va, Y_va, y_va = Xall[:, m_tr:m_tr+m_va],   Y[:, m_tr:m_tr+m_va],   ylab[m_tr:m_tr+m_va]
X_te, Y_te, y_te = Xall[:, m_tr+m_va:],       Y[:, m_tr+m_va:],       ylab[m_tr+m_va:]

# ----- fonctions -----
def sigma(z): return 1/(1+np.exp(-z))
def sigma_prim(z): return np.exp(-z)/((1+np.exp(-z))**2)
def softmax(z):
    e = np.exp(z - np.max(z, axis=0)); return e/np.sum(e, axis=0)
def model_predict(X, W2, W3, b2, b3):
    return softmax(W3 @ sigma(W2@X + b2) + b3)
def cost(Y, Yp, W2, W3, lam):
    mm = Y.shape[1]
    return (-1/mm)*np.sum(Y*np.log(Yp+1e-12)) + (lam/(2*mm))*(np.sum(W2**2)+np.sum(W3**2))
def compute_grad(x, y, W2, W3, b2, b3, lam):
    mm = x.shape[1]; A1 = x; Z2 = W2@A1+b2; A2 = sigma(Z2); A3 = softmax(W3@A2+b3)
    d3 = A3 - y; d2 = (W3.T@d3)*sigma_prim(Z2)
    return ((1/mm)*d2@A1.T + (lam/mm)*W2, (1/mm)*d3@A2.T + (lam/mm)*W3,
            (1/mm)*np.sum(d2,axis=1).reshape((W2.shape[0],1)),
            (1/mm)*np.sum(d3,axis=1).reshape((W3.shape[0],1)))
def accuracy(X, W2, W3, b2, b3, labels):
    return 100*np.mean(np.argmax(model_predict(X,W2,W3,b2,b3),axis=0) == labels)

def entrainer(n2, Niter, alpha, lam, seed=1):
    np.random.seed(seed)
    W2 = np.random.randn(n2, n1)*0.1; b2 = np.zeros((n2,1))
    W3 = np.random.randn(n3, n2)*0.1; b3 = np.zeros((n3,1))
    for j in range(Niter):
        g = compute_grad(X_tr, Y_tr, W2, W3, b2, b3, lam)
        W2-=alpha*g[0]; W3-=alpha*g[1]; b2-=alpha*g[2]; b3-=alpha*g[3]
    return W2, W3, b2, b3

# ----- tableau d'hyperparamètres (alpha et lambda fixés) -----
alpha, lam = 0.3, 0.5
best_acc, best = 0, None
print(f"{'n2':<4} {'Niter':<6} {'accV(%)'}")
for n2 in [10, 20, 30]:
    for Niter in [300, 600, 900]:          # remplacer par 1500/3000/4500 à l'examen
        W2,W3,b2,b3 = entrainer(n2, Niter, alpha, lam)
        a = accuracy(X_va, W2, W3, b2, b3, y_va)
        print(f"{n2:<4} {Niter:<6} {a:.2f}")
        if a > best_acc:
            best_acc, best, best_hp = a, (W2,W3,b2,b3), (n2,Niter)

# ----- test final avec les meilleurs hyperparamètres -----
W2,W3,b2,b3 = best
print("\nMeilleurs hyperparamètres :", best_hp)
print("Accuracy TEST :", round(accuracy(X_te, W2, W3, b2, b3, y_te), 2), "%")

### C. Régression — réseau de neurones à sortie linéaire (examen 2025 Q1–3)

Sortie `a3 = z3` (pas d'activation), coût = erreur quadratique moyenne (MSE).
On découpe en 900 train / 100 test, on normalise dans [0,1], on entraîne, on trace le coût,
on calcule la MSE de test.


In [ ]:
# ----- données synthétiques (14 variables, type examen 2025) -----
np.random.seed(1)
n1 = 14; m_tr = 900; m_te = 100; m_total = m_tr + m_te
Xall = np.random.rand(n1, m_total)
w_vrai = np.random.randn(n1)
yall = (w_vrai @ Xall + 0.1*np.random.randn(m_total)).reshape(1, m_total)*10 + 50  # notes ~[?]

# découpage 900/100
X = Xall[:, :m_tr];  y = yall[:, :m_tr]
X_test = Xall[:, m_tr:];  y_test = yall[:, m_tr:]

# normalisation [0,1] (paramètres du train)
mn = np.min(X, axis=1); mx = np.max(X, axis=1)
X      = ((X.T      - mn)/(mx-mn)).T
X_test = ((X_test.T - mn)/(mx-mn)).T

# ----- paramètres imposés -----
m = m_tr; n2 = 20; alpha = 0.2; Niter = 800

def sigma(z): return 1/(1+np.exp(-z))
def sigma_prim(z): return np.exp(-z)/((1+np.exp(-z))**2)
def model_predict(X, W2, W3, b2, b3):
    return W3 @ sigma(W2@X + b2) + b3          # sortie LINÉAIRE
def cost(y, y_pred):
    return np.linalg.norm(y - y_pred)**2 / np.size(y)   # MSE
def compute_grad(x, y, W2, W3, b2, b3):       # backprop fournie à l'examen
    A1 = x; Z2 = W2@A1+b2; A2 = sigma(Z2); A3 = W3@A2+b3
    d3 = A3 - y; d2 = (W3.T@d3)*sigma_prim(Z2)
    return ((1/m)*d2@A1.T, (1/m)*d3@A2.T,
            (1/m)*np.sum(d2,axis=1).reshape((n2,1)), (1/m)*np.sum(d3,axis=1).reshape((1,1)))

# ----- init (n3 = 1) -----
np.random.seed(2)
W2 = np.random.randn(n2, n1); b2 = np.random.randn(n2, 1)
W3 = np.random.randn(1, n2);  b3 = np.random.randn(1, 1)

# ----- entraînement -----
Cost = np.zeros(Niter)
for j in range(Niter):
    Cost[j] = cost(y, model_predict(X,W2,W3,b2,b3))
    g = compute_grad(X, y, W2, W3, b2, b3)
    W2-=alpha*g[0]; W3-=alpha*g[1]; b2-=alpha*g[2]; b3-=alpha*g[3]

print("MSE test :", round(cost(y_test, model_predict(X_test,W2,W3,b2,b3)), 4))
plt.plot(Cost); plt.xlabel("itérations"); plt.ylabel("MSE"); plt.title("C - coût (train)"); plt.show()

### D. Validation croisée — moyenne de 5 modèles (examen 2025 Q4–5)

À partir des 900 données d'entraînement, on crée 5 découpages (4/5 train, 1/5 test interne).
On entraîne 5 modèles, et la prédiction finale est la **moyenne** des 5. On évalue sur les 100
données test jamais utilisées, et on compare à la MSE des modèles pris séparément.

*Dimensions (Q4) :* avec 900 données et 5 plis, chaque `Xtrain,k` est `(n1, 720)` et chaque
`Xtest,k` est `(n1, 180)` ; `ytrain,k` est `(1, 720)`, `ytest,k` est `(1, 180)`.


In [ ]:
# on réutilise un dataset de régression (14 variables, 900 train + 100 test)
np.random.seed(1)
n1 = 14; m_tr = 900; m_te = 100; m_total = m_tr + m_te
Xall = np.random.rand(n1, m_total)
w_vrai = np.random.randn(n1)
yall = (w_vrai @ Xall + 0.1*np.random.randn(m_total)).reshape(1, m_total)*10 + 50

X_train = Xall[:, :m_tr].copy(); y_train = yall[:, :m_tr].copy()
mn = np.min(X_train, axis=1); mx = np.max(X_train, axis=1)
X_train = ((X_train.T - mn)/(mx-mn)).T
X_test  = ((Xall[:, m_tr:].T - mn)/(mx-mn)).T;  y_test = yall[:, m_tr:]

def sigma(z): return 1/(1+np.exp(-z))
def sigma_prim(z): return np.exp(-z)/((1+np.exp(-z))**2)
def cost(y, yp): return np.linalg.norm(y - yp)**2 / np.size(y)

def train_reg(Xtr, ytr, n2=20, alpha=0.2, Niter=800, seed=3):
    np.random.seed(seed)
    n1 = Xtr.shape[0]; mm = Xtr.shape[1]
    W2 = np.random.randn(n2, n1); b2 = np.random.randn(n2, 1)
    W3 = np.random.randn(1, n2);  b3 = np.random.randn(1, 1)
    for j in range(Niter):
        Z2 = W2@Xtr+b2; A2 = sigma(Z2); A3 = W3@A2+b3
        d3 = A3 - ytr; d2 = (W3.T@d3)*sigma_prim(Z2)
        W2 -= alpha*((1/mm)*d2@Xtr.T); W3 -= alpha*((1/mm)*d3@A2.T)
        b2 -= alpha*((1/mm)*np.sum(d2,axis=1).reshape((n2,1)))
        b3 -= alpha*((1/mm)*np.sum(d3,axis=1).reshape((1,1)))
    return W2, W3, b2, b3

def predire(mdl, X):
    W2, W3, b2, b3 = mdl
    return W3 @ sigma(W2@X + b2) + b3

# ----- 5 plis -----
K = 5; fold = m_tr // K           # 180
modeles = []; mse_separes = []
for k in range(K):
    idx_test  = np.arange(k*fold, (k+1)*fold)
    idx_train = np.setdiff1d(np.arange(m_tr), idx_test)
    mdl = train_reg(X_train[:, idx_train], y_train[:, idx_train])
    modeles.append(mdl)
    pred_k = predire(mdl, X_train[:, idx_test])
    mse_separes.append(cost(y_train[:, idx_test], pred_k))

# ----- modèle final = moyenne des 5, évalué sur les 100 test -----
pred_ens = np.mean([predire(mdl, X_test) for mdl in modeles], axis=0)
print("MSE de chaque modèle (pli interne) :", [round(e,3) for e in mse_separes])
print("MSE moyenne des modèles séparés     :", round(np.mean(mse_separes), 4))
print("MSE du modèle final (ensemble) test :", round(cost(y_test, pred_ens), 4))

### E. Régression logistique régularisée (session 2024 Q5)

La régression logistique régularisée est exactement le réseau de neurones **sans couche cachée** :
`a = softmax(W·x + b)`. On fait varier λ et on lit l'accuracy de validation.
(C'est la même mécanique que le scénario B, mais sans `W2/b2`.)


In [ ]:
# ----- données (4 classes, type météo) -----
np.random.seed(0)
n1, n3 = 7, 4; m_total = 600
centres = np.random.randn(n3, n1)*3
Xall = np.zeros((n1, m_total)); ylab = np.zeros(m_total, dtype=int)
for i in range(m_total):
    c = i % n3; ylab[i] = c; Xall[:, i] = centres[c] + np.random.randn(n1)
Xall, ylab = shuffle(Xall.T, ylab, random_state=42); Xall = Xall.T
Y = np.zeros((n3, m_total))
for i in range(m_total): Y[ylab[i], i] = 1.0

m_tr, m_va = 360, 120
mn = np.min(Xall[:, :m_tr], axis=1); mx = np.max(Xall[:, :m_tr], axis=1)
Xall = ((Xall.T - mn)/(mx-mn+1e-12)).T
X_tr, Y_tr, y_tr = Xall[:, :m_tr],         Y[:, :m_tr],         ylab[:m_tr]
X_va, Y_va, y_va = Xall[:, m_tr:m_tr+m_va], Y[:, m_tr:m_tr+m_va], ylab[m_tr:m_tr+m_va]
X_te, Y_te, y_te = Xall[:, m_tr+m_va:],     Y[:, m_tr+m_va:],     ylab[m_tr+m_va:]

def softmax(z):
    e = np.exp(z - np.max(z, axis=0)); return e/np.sum(e, axis=0)

def logreg_train(X, Y, alpha=0.3, Niter=500, lam=0.0, seed=1):
    np.random.seed(seed)
    n3, n1 = Y.shape[0], X.shape[0]; mm = X.shape[1]
    W = np.random.randn(n3, n1)*0.1; b = np.zeros((n3, 1))
    for j in range(Niter):
        A = softmax(W@X + b)
        d = A - Y
        dW = (1/mm)*d@X.T + (lam/mm)*W      # régularisation sur W
        db = (1/mm)*np.sum(d, axis=1).reshape((n3, 1))
        W -= alpha*dW; b -= alpha*db
    return W, b

def logreg_acc(X, W, b, labels):
    return 100*np.mean(np.argmax(softmax(W@X + b), axis=0) == labels)

print(f"{'lambda':<8} {'accV(%)'}")
best = (0, None, None)
for lam in [0.3, 1, 10, 20]:
    W, b = logreg_train(X_tr, Y_tr, alpha=0.3, Niter=500, lam=lam)
    a = logreg_acc(X_va, W, b, y_va)
    print(f"{lam:<8} {a:.2f}")
    if a > best[0]: best = (a, (W, b), lam)

W, b = best[1]
print("\nMeilleur lambda :", best[2])
print("Accuracy TEST :", round(logreg_acc(X_te, W, b, y_te), 2), "%")

### F. K-means (TP8)

Trois étapes : assignation au centroïde le plus proche, recalcul des centroïdes, répétition.


In [ ]:
# ----- données 2D synthétiques (3 amas) -----
np.random.seed(5)
X = np.vstack([np.random.randn(50,2)+[0,0],
               np.random.randn(50,2)+[5,5],
               np.random.randn(50,2)+[5,0]])

def findClosestCentroids(X, centroids):
    K = centroids.shape[0]
    idx = np.zeros(X.shape[0], dtype=int)
    for i in range(X.shape[0]):
        idx[i] = np.argmin(np.sum((X[i, :] - centroids)**2, axis=1))
    return idx

def computeCentroids(X, idx, K):
    m, n = X.shape
    centroids = np.zeros((K, n))
    for k in range(K):
        pts = X[idx == k]
        if len(pts) > 0:
            centroids[k, :] = np.mean(pts, axis=0)
    return centroids

def kMeansInitCentroids(X, K):
    return X[np.random.permutation(X.shape[0])[:K], :]

K = 3
centroids = kMeansInitCentroids(X, K)
for it in range(10):
    idx = findClosestCentroids(X, centroids)
    centroids = computeCentroids(X, idx, K)

print("Centroïdes finaux :\n", np.round(centroids, 2))
plt.scatter(X[:,0], X[:,1], c=idx, s=15)
plt.scatter(centroids[:,0], centroids[:,1], c="k", marker="X", s=120)
plt.title("F - K-means"); plt.show()

### G. PCA (TP8)

Normaliser, calculer les composantes principales (SVD de la covariance), projeter, reconstruire.


In [ ]:
np.random.seed(6)
X = np.random.randn(100, 2) @ np.array([[2., 1.], [1., 1.]])   # données corrélées

def featureNormalize(X):
    mu = np.mean(X, axis=0)
    X_norm = X - mu
    sigma = np.std(X_norm, axis=0, ddof=1)
    X_norm = X_norm / sigma
    return X_norm, mu, sigma

def pca(X):
    m, n = X.shape
    Sigma = (1/m) * X.T @ X          # matrice de covariance
    U, S, V = np.linalg.svd(Sigma)   # U = vecteurs propres, S = valeurs singulières
    return U, S

def projectData(X, U, K):
    return X @ U[:, :K]              # projection sur les K premières composantes

def recoverData(Z, U, K):
    return Z @ U[:, :K].T            # reconstruction approchée

X_norm, mu, sigma = featureNormalize(X)
U, S = pca(X_norm)
K = 1
Z = projectData(X_norm, U, K)
X_rec = recoverData(Z, U, K)

print("Composantes principales U :\n", np.round(U, 3))
print("Erreur de reconstruction (K=1) :", round(np.mean((X_norm - X_rec)**2), 4))
plt.scatter(X_norm[:,0], X_norm[:,1], s=12, label="données")
plt.scatter(X_rec[:,0],  X_rec[:,1],  s=12, label="reconstruit (K=1)")
plt.legend(); plt.axis("equal"); plt.title("G - PCA"); plt.show()

### H. Biais-variance — courbes d'apprentissage et de validation (TP6)

Régression linéaire régularisée via `optimize.minimize`. La **courbe d'apprentissage** trace l'erreur
train/validation en fonction du nombre d'exemples (diagnostic biais/variance). La **courbe de validation**
trace l'erreur en fonction de λ pour le choisir.


In [ ]:
# ----- données 1D synthétiques -----
np.random.seed(7)
X    = np.linspace(-5, 5, 12).reshape(-1, 1)
y    = 0.5*X.ravel()**2 - 2*X.ravel() + np.random.randn(12)*2
Xval = np.linspace(-4, 4, 8).reshape(-1, 1)
yval = 0.5*Xval.ravel()**2 - 2*Xval.ravel() + np.random.randn(8)*2
m = y.size

def linearRegCostFunction(X, y, theta, lambda_=0.0):
    m = y.size
    h = X @ theta
    temp = theta.copy(); temp[0] = 0          # on ne régularise pas le biais theta_0
    J = (1/(2*m))*np.sum((h-y)**2) + (lambda_/(2*m))*np.sum(temp**2)
    grad = (1/m)*(X.T @ (h-y)) + (lambda_/m)*temp
    return J, grad

def trainLinearReg(costf, X, y, lambda_=0.0, maxiter=200):
    init = np.zeros(X.shape[1])
    res = optimize.minimize(lambda t: costf(X, y, t, lambda_), init,
                            jac=True, method='TNC', options={'maxiter': maxiter})
    return res.x

def learningCurve(X, y, Xval, yval, lambda_=0):
    m = y.size
    error_train = np.zeros(m); error_val = np.zeros(m)
    for i in range(1, m+1):
        theta = trainLinearReg(linearRegCostFunction, X[:i], y[:i], lambda_)
        error_train[i-1], _ = linearRegCostFunction(X[:i], y[:i], theta, 0)   # erreur avec lambda=0
        error_val[i-1],   _ = linearRegCostFunction(Xval, yval, theta, 0)
    return error_train, error_val

def validationCurve(X, y, Xval, yval):
    lambda_vec = [0, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]
    et = np.zeros(len(lambda_vec)); ev = np.zeros(len(lambda_vec))
    for i, l in enumerate(lambda_vec):
        theta = trainLinearReg(linearRegCostFunction, X, y, l)
        et[i], _ = linearRegCostFunction(X, y, theta, 0)
        ev[i], _ = linearRegCostFunction(Xval, yval, theta, 0)
    return lambda_vec, et, ev

# ajout de la colonne de 1 (terme de biais)
Xa    = np.concatenate([np.ones((m, 1)), X], axis=1)
Xval_a = np.concatenate([np.ones((yval.size, 1)), Xval], axis=1)

et, ev = learningCurve(Xa, y, Xval_a, yval, lambda_=0)
lv, et2, ev2 = validationCurve(Xa, y, Xval_a, yval)
print("lambda optimal (min erreur validation) :", lv[int(np.argmin(ev2))])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(np.arange(1, m+1), et, label="train"); ax[0].plot(np.arange(1, m+1), ev, label="validation")
ax[0].set_title("Courbe d'apprentissage"); ax[0].set_xlabel("nb d'exemples"); ax[0].legend()
ax[1].plot(lv, et2, '-o', label="train"); ax[1].plot(lv, ev2, '-o', label="validation")
ax[1].set_title("Courbe de validation (lambda)"); ax[1].set_xlabel("lambda"); ax[1].legend()
plt.show()

## 4. Aide-mémoire express (quoi copier selon le sujet)

| Le sujet demande… | Va voir… |
|---|---|
| Lire/nettoyer un CSV, encoder des catégories | 2.1 |
| One-hot encoding | 2.2 |
| Mélanger les données | 2.3 |
| Découper en 2 ou 3 jeux | 2.4 |
| Normaliser entre 0 et 1 | 2.5 |
| Classifier 2 classes | A |
| Classifier ≥ 3 classes + remplir un tableau (n2, Niter) | B |
| Prédire une valeur numérique (régression) | C |
| « Construire 5 modèles / moyenne / validation croisée » | D |
| « Régression logistique régularisée », faire varier λ | E |
| Regrouper sans labels (clustering) | F |
| Réduire la dimension / compresser | G |
| Courbes d'apprentissage, choisir λ, biais/variance | H |
| Question théorique (dimensions, shuffle, 3 jeux…) | Section 1 |

**Réflexes à l'examen :**
1. Régression → sortie linéaire (`a3 = z3`) + coût MSE + `compute_grad_reg`.
2. Classification → softmax + entropie croisée + `compute_grad_classif` ; décision par `argmax`.
3. `delta_3 = a3 − y` dans les deux cas (avec entropie croisée).
4. Normalisation : min/max **calculés sur le train**, appliqués au test.
5. Régularisation : `+ (λ/m)·W` sur les poids (pas les biais).
6. Hyperparamètres choisis sur la **validation**, jamais sur le test.
